# COMPUTACIÓN CUÁNTICA - OPTATIVA - GRADOS ING. INFORMÁTICA Y SOFTWARE

# _Notebook_ sobre puertas y circuitos de 1 cúbit

## Entorno de trabajo 

In [ ]:
# No olvidar que en "google colab" hay que instalar qiskit
# Instala la última versión disponible en collab (2.1)
try:
    import google.colab
    print("In colab, let's install things...")
    #
    !pip install qiskit==2.1.2 qiskit[visualization] qiskit-aer qiskit-ibm-runtime qiskit-optimization matplotlib numpy scipy sympy pandas
except ImportError:
    print("NOT in colab")

In [ ]:
# Print package versions
import importlib
packages = [
    "qiskit",
    "qiskit_aer",
    "qiskit_ibm_runtime",
    "qiskit_optimization",
]
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:25s} : {mod.__version__}")
    except ImportError:
        print("%-25s: %s"%(pkg, "not installed"))


## Paquetería

In [ ]:
import numpy as np
import scipy.linalg as la
from IPython.display import display, Markdown, Latex, HTML
import matplotlib.pyplot as plt
import copy 

from qiskit.visualization import array_to_latex
from qiskit.visualization import plot_histogram, plot_bloch_multivector, plot_state_qsphere, array_to_latex
from qiskit.quantum_info import Statevector, schmidt_decomposition
from qiskit.circuit import QuantumRegister, ClassicalRegister,QuantumCircuit
from qiskit import transpile
from qiskit_aer import AerSimulator

## Utilería  

In [ ]:
# Utilería acumulada

# Definimos una función para visualizar el estado y la esfera a la vez
# Permite escalar la(s) esferas(s) con 'inches' pulgadas, si este parámetro
# no se pone, se dibuja al tamaño por defecto
# Si se pone 0, no se dibuja la esfera
def my_display_state_vector(psi, inches=None, prefix=r"|\psi\rangle = "):
    d_latex = psi.draw('latex', prefix=prefix)
    display(d_latex)
    #
    d_bloch = psi.draw('bloch')
    if isinstance(inches, (int,float)):
        if inches > 0:
            d_bloch.set_size_inches(inches, inches, forward=True)
            display(d_bloch)
    else:
        display(d_bloch)



<a id='nb03_sec1'></a>
## Iniciar un circuito a partir del vector de estado

En este ejercicio vamos a constuir un circuito de un qubit meramente inicializado con un valor arbitrario y realizaremos su simulación

In [ ]:
# Iniciando un simple circuito sin puertas

# Vamos a generar un vector de estado de un qubit a un estado aleatorio
rm = np.random.rand(2,1) + np.random.rand(2,1)*1j
rm = rm/la.norm(rm) # Normalizar 
initial_state = Statevector(rm)

# Vamos a mostar este estado con 2 decimales
# Recordar que los States vectors se admiten no normalizados, 
# otra cosa es que no sean válidos
my_display_state_vector(Statevector([np.round(initial_state[0],2), np.round(initial_state[1],2)]), 2.5) 

In [ ]:
# Creamos un circuito cuántico de 1 qubit
# con una mera inicialización y lo mostramos
qc = QuantumCircuit(1)
qc.initialize(initial_state)

# Mostramos el circuito en estilo matplotlib (mpl)
fig = qc.draw('mpl', style="iqp")
display(fig)


Obsérvese que el método `QuantumCircuit.draw('mpl')` genera una figura de Matplotlib, y la hemos almacenado en una variable, por lo que la podemos usar con los métodos típicos de Matplotlib como el `savefig()` para guardar la figura en un fichero

In [ ]:
# Salvemos la figura con el circuito en un fichero y comprobemos
# que se ha generado

filename = "/tmp/Fig_circuit_initial_state.png"
fig.savefig(filename)
!ls -l $filename # Chequear que el fichero se ha generado, escapando al shell

<div style="border:1pt black solid; padding:3pt;">
    
<b>Ejercicio:</b>
    
Lee en la documentación en que otros formatos se puede _dibujar_ el ciruito (texto, Latex,...) y pruébalos

Puedes acceder a la documentación en la [referencia de Qiskit online](https://quantum.cloud.ibm.com/docs/es/api/qiskit/qiskit.circuit.QuantumCircuit#draw), o ejecutando el comando de ayuda sobre el método ```draw``` de la clase ```QuantumCircuit```:  

```python
     help(QuantumCircuit.draw)
```

</div>

In [ ]:
# Qiskit permite dibujar el mismo circuito como texto, como código LaTeX o como figura.
# El formato de texto es útil en una terminal; latex_source devuelve el código fuente
# LaTeX (que después puede compilarse), y mpl genera una figura de Matplotlib.
print('Formato texto:')
print(qc.draw(output='text'))

print('\nCódigo fuente LaTeX:')
print(qc.draw(output='latex_source'))

print('Figura Matplotlib:')
display(qc.draw(output='mpl', style='iqp'))


**Simulación de un circuito**

Para simular un circuito con el simulador _Aer_ (simulador incluido en el ecosistema Qiskit) seguiremos los siguientes pasos:
- Preparar una **instancia del simulador** Aer (hay varios tipos de simulación disponibles)
- **Transpilar** el circuito (operación análoga a la compilación, el circuito se optimiza y se transforma en un circuito equivalente capaz de ser ejecutado en el HW cuántico objetivo)
- **Ejecutar** (puesto que el resultado de las medidas es probabilístico, generalmente la simulación se repite un número de veces, que denominamos _shots_)
- Obtener el **resultado**:
    - En estos primeros ejemplos usaremos un _God mode_ que permite ver el estado resultante, para ello se añade una _sonda_ (_probe_) en la cual se almacena el estado final de la simulación
    - En la práctica solo tenemos el conteo de las mediciones posibles, sobre las que estimaremos las probabilidades

In [ ]:
# Para poder observar el estado final de la simulación (cosa que
# en un computador cuántico real es imposible) añadimos una "sonda"
# de simulación

qcnew = qc.copy()   # Hacemos una copia del cicuito original para añadir la "sonda"
                    # ¿Por qué hacer una copia, no asignar simplemente?
qcnew.save_statevector()


<div style="border:1pt black solid; padding:3pt;">
    
<b>Ejercicio:</b>

Dibuja el nuevo circuito, ¿qué ha pasado?

</div>

In [ ]:
# La sonda save_statevector aparece como una instrucción adicional en el dibujo.
# Es una instrucción de Aer para guardar el estado durante la simulación;
# no es una puerta física y no transforma el estado del qubit.
print(qcnew.draw(output='text'))
display(qcnew.draw(output='mpl', style='iqp'))
display(Markdown('La caja `save_statevector` es una sonda de simulación. Guarda una copia del vector de estado en ese punto del circuito, sin alterar el estado cuántico.'))


In [ ]:
# Generamos una instancia del simulador;
# con "method" le especificamos el método de simulación
simulador = AerSimulator(method = 'statevector')

# También podemos llamar primero a la clase del simulador sin
# pasarle argumentos y después añadir las opciones:
# simulador = AerSimulator()
# sim.set_options(method = 'statevector')

# Transpilamos (para un ejemplo tan sencillo ni siquiera haría falta
qcnewt = transpile(qcnew, backend = simulador)

# Ejecutamos la simulación, para ver el estado sería 
# suficiente con 1 shot, pero pongamos más
job = simulador.run(qcnewt, shots = 1)
result = job.result()

In [ ]:
# Como pusimos la sonda, podemos coger y observar el estado cuántico final
# aunque ¡¡en la realidad física esto no es posible!!
final_state = result.get_statevector()

# Veamos cómo es
my_display_state_vector(final_state, 2)

In [ ]:
# Si el simulador simula bien, debemos haber obtenido el
# mismo estado que inicalmente
assert(final_state == initial_state)

In [ ]:
# Del objeto resultado, también podemos ver la probabilidad
# de medir un |0> o un |1>
# (aunque esto lo haremos luego poniendo medidores!!)

# Como no hemos puesto medidores éste es un resultado, ideal,
# que la rutina obtiene a partir del cuadrado de las amplitudes de |0> y |1>

# Observa que la estructura devuelta por la rutina get_counts()
# es un diccionario

counts_ideal = result.get_counts(0)
display(counts_ideal)
print("\nTipo de dato devuelto: " + str(type(counts_ideal)))

In [ ]:
# Comprobemos que el resultado del simulador coinciden
# con el cuadrado de las amplitudes
p0 = np.abs(initial_state[0])**2
p1 = np.abs(initial_state[1])**2
print(" p0={0}\n p1={1}".format(p0,p1))

### Estado final para vagos 

In [ ]:
# Se puede obtener directamente el estado final de un circuito
# sin recurrir a la simulación, a través de la clase statevector
# de diferentes maneras:

# Método Statevector.from_instruction()
state_from_circuit = Statevector.from_instruction(qc) # Asume que el estadof final es cero
display(state_from_circuit.draw('latex', prefix=r'|\Psi_{\text{from circuit}}\rangle = '))
assert(state_from_circuit.equiv(final_state))

# Método Statevector.evolve()
one = Statevector.from_int(1, dims=2) #Este establece el valor inicial, por ejemplo |1>
state_from_one = one.evolve(qc)
display(state_from_one.draw('latex', prefix=r'|\Psi_{\text{from one}}\rangle = '))
assert(state_from_one.equiv(final_state))

# Y así podemos obtener el diccionario de probabilidades
prob_from_circuit = state_from_circuit.probabilities_dict()
prob_from_one = state_from_one.probabilities_dict()
display(Latex(r"Probabilidades para $|\Psi_{\text{from circuit}}\rangle$:"), prob_from_circuit)
display(Latex(r"Probabilidades para $|\Psi_{\text{from one}}\rangle$:"), prob_from_one)


## Puertas de un qubit

In [ ]:
# Definamos alguna utilería

# Función para simular un cirucito dado:
# devuelve esta terna (estado final, cuenta, resultado del job)
def my_get_circuit_state(qcirc_in, shots=1):
    # Instancia del simulador
    sim = AerSimulator(method = "statevector")
    
    # Hacemos una copia ya que vamos a cambiar el objeto circuito
    qcirc = qcirc_in.copy()
    
    # Ponemos el "probe" para el vector de estado al final
    # del cirucito dado
    # ojo! el circuito no debe de tener un "save_statevector" ya!
    qcirc.save_statevector()
    
    # Hacemos una simulación y obtenemos los resultados
    qobj = transpile(qcirc, sim)
    result = sim.run(qobj, shots=shots).result()
    state = result.get_statevector()
    counts_ideal = result.get_counts()
    
    return state, counts_ideal, result


### Puerta X

Veamos ahora un ejemplo de un circuito con la puerta $X$ para ver su acción

In [ ]:
# Creamos el circuito de 1 solo qubit
circuit_X = QuantumCircuit(1)

# Añadimos la puerta X al primer (y único) qúbit
# Nótese que por defecto el estado inicial si no se indica es |0>
circuit_X.x(0)

# Podemos dibujarlo en modo texto
#print(circuit_X.draw(output='text'))

# Podemos dibujarlo como figura de matplolib
fig = circuit_X.draw(output='mpl',  style="iqp") # mpl es de Matplolib

# Podemos guardar la figura en fichero
#fig.savefig("/tmp/Fig_circuit_X.png")
# Invocar al sistema, comprobar que el fichero se ha creado
#!ls -l "/tmp/Fig_circuit_X.png"

# Si queremos mostramos el circuito
display(fig)

In [ ]:
# Usemos nuestra función de utilería que simula y nos da el estado final
state, p_ideal, r = my_get_circuit_state(circuit_X)

# Visualicemos el estado y las probabilidades
my_display_state_vector(state,2.2, r"\Psi_{out X}=")
display(p_ideal)


### Puerta H

Veamos ahora un ejemplo de un circuito con la puerta $H$ para ver su acción

In [ ]:
# Creamos el circuito de 1 solo qubit
circuit_H = QuantumCircuit(1)

# Añadimos la puerta X al primer (y único) qúbit
# Nótese que por defecto el estado inicial si no se indica es |0>
circuit_H.h(0)

# Podemos dibujarlo en modo texto
#print(circuit_H.draw(output='text'))

# Podemos dibujarlo como figura de matplolib
fig = circuit_H.draw(output='mpl',  style="iqp") # mpl es de Matplolib

# Podemos guardar la figura en fichero
#Fig.savefig("/tmp/Fig_circuit_H.png")
# Invocar al sistema, comprobar que el fichero se ha creado
#!ls -l "/tmp/Fig_circuit_H.png"

# Si queremos mostramos el circuito
display(fig)

In [ ]:
# Usemos nuestra función de utilería que simula y nos da el estado final
state, p_ideal, r = my_get_circuit_state(circuit_H)

# Visualicemos el estado y las probabilidades
my_display_state_vector(state,2.2, r"\Psi_{out H}=")
display(p_ideal)

Como podemos ver, ahora aproximadamente la mitad de las veces obtenemos $| 0 \rangle$ y la otra mitad obtenemos $| 1 \rangle$. Esto es debido a que 
$$
H |0  \rangle = \frac{1}{\sqrt{2}} \left(| 0 \rangle + | 1 \rangle \right)
$$
Recordemos que la probabilidad de medir cada estado es igual al módulo cuadrado de su amplitud (del coeficiente que acompaña al estado)

### Circuito de un qubit con varias puertas

Veamos ahora un ejemplo de un circuito con varias puertas
Y vamos a poner varios "probes" para ver los valores intermedios del vector de estado

In [ ]:
# Creamos el circuito de 1 solo qubit
circuit_M = QuantumCircuit(1)

# Añadimos la puerta X al primer (y único) qúbit
# Nótese que por defecto el estado inicial si no se indica es |0>
circuit_M.y(0)
circuit_M.save_statevector(label='after_Y')    # Probe para statevector intermedio
circuit_M.h(0)
circuit_M.save_statevector(label='after_H')    # Probe para statevector intermedio
circuit_M.s(0)
circuit_M.save_statevector(label='after_S')    # Probe para statevector intermedio

# Podemos dibujarlo en modo texto
#print(circuit_M.draw(output='text'))

# Podemos dibujarlo como figura de matplolib
fig = circuit_M.draw(output='mpl',  style="iqp") # mpl es de Matplolib

# Si queremos mostramos el circuito
display(fig)

In [ ]:
# Usemos nuestra función de utilería que simula y nos da el estado final
state, p_ideal, result = my_get_circuit_state(circuit_M)

# Visualicemos el estado y las probabilidades finales
my_display_state_vector(state,2.2, r"\Psi_{out}=")
display(p_ideal)

In [ ]:
# Veamos los estados intermedios
# Esto es un 'dirty trick'
result_state_0 = result.data(0)['after_Y']
result_state_1 = result.data(0)['after_H']
result_state_2 = result.data(0)['after_S']

my_display_state_vector(result_state_0, 0, r"\Psi_{after Y}=")
my_display_state_vector(result_state_1, 0, r"\Psi_{after H}=")
my_display_state_vector(result_state_2, 0, r"\Psi_{after S}=")

<div style="border:1pt black solid; padding:3pt;">
    
<b>Ejercicio:</b>

Construye un circuito que partiendo del estado inicial $\displaystyle \frac{\sqrt{2}}{2}(|0\rangle - e^{\large i\frac{\pi}{3}} |1\rangle)$ aplique las puertas necesarias para la siguiente secuencia de transformaciones, en este orden:

 1. rotación en torno al eje Z de $\displaystyle \frac{5\pi}{3}$ radianes
 2. intercambio de $|1\rangle$ con $|0\rangle$
 3. aplicación de la primera raíz cuarta de $Z$: $\displaystyle \sqrt[\large 4]{Z}$
 4. una rotación $\pi$ radianes en torno al eje eje $\hat{x}+\hat{y}$ (es decir, el eje diagonal situado a 45◦ entre el eje X y el eje Z)

Introduce sondas a la entrada de cada puerta y a la salida final del circuito.

Realiza la simulación ```Aer``` del vector de estados, imprimiendo el estado resultante en la entrada de cada puerta y al final del circuito, así como las probabilidades ideales de medir $|0\rangle$ y $|1\rangle$. 

Comprueba que los resultados obtenidos son acordes con lo que se espera (o sea, calcula matemáticamente a lápiz y papel cuál deben ser las salidas y cotéjalas con el resultado del simulador Aer).

</div>

In [ ]:
# Estado inicial: (|0> - exp(i*pi/3)|1>)/sqrt(2).
# Las sondas guardan el estado a la entrada de cada puerta y al final.
theta = 5 * np.pi / 3
psi0 = np.array([1, -np.exp(1j * np.pi / 3)], dtype=complex) / np.sqrt(2)

qc_practica4 = QuantumCircuit(1)
qc_practica4.initialize(psi0, 0)
qc_practica4.save_statevector(label='entrada_Rz')
qc_practica4.rz(theta, 0)
qc_practica4.save_statevector(label='entrada_X')
qc_practica4.x(0)  # Intercambia |0> y |1>
qc_practica4.save_statevector(label='entrada_T')
qc_practica4.t(0)  # Primera raíz cuarta de Z: diag(1, exp(i*pi/4))
qc_practica4.save_statevector(label='entrada_Rxy')

# Interpreto literalmente el eje x+y: n=(x+y)/sqrt(2), en el plano XY
# (45 grados entre X e Y). La mención del enunciado a X y Z parece una errata.
# R_n(pi) = exp[-i*pi*(X+Y)/(2*sqrt(2))]. Su matriz es -i(X+Y)/sqrt(2).
pauli_x = np.array([[0, 1], [1, 0]], dtype=complex)
pauli_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
rotation_xy = -1j * (pauli_x + pauli_y) / np.sqrt(2)
qc_practica4.unitary(rotation_xy, [0], label='R_xy(pi)')
qc_practica4.save_statevector(label='salida_final')
display(qc_practica4.draw(output='mpl', style='iqp'))

# Simulación Aer
simulador_p4 = AerSimulator(method='statevector')
qc_practica4_t = transpile(qc_practica4, backend=simulador_p4)
resultado_p4 = simulador_p4.run(qc_practica4_t, shots=1).result()

# Cálculo analítico mediante las matrices de cada transformación.
rz = np.diag([np.exp(-1j * theta / 2), np.exp(1j * theta / 2)])
matriz_x = pauli_x
matriz_t = np.diag([1, np.exp(1j * np.pi / 4)])
estados_esperados = {
    'entrada_Rz': psi0,
    'entrada_X': rz @ psi0,
    'entrada_T': matriz_x @ rz @ psi0,
    'entrada_Rxy': matriz_t @ matriz_x @ rz @ psi0,
    'salida_final': rotation_xy @ matriz_t @ matriz_x @ rz @ psi0,
}
for etiqueta, esperado in estados_esperados.items():
    obtenido = np.asarray(resultado_p4.data(0)[etiqueta])
    assert np.allclose(obtenido, esperado), f'No coincide el estado {etiqueta}'
    print(f'\nEstado {etiqueta}:')
    my_display_state_vector(
        Statevector(obtenido), inches=0,
        prefix=rf'|\psi_{{{etiqueta}}}\rangle = '
    )

# Las probabilidades ideales son los módulos cuadrados de las amplitudes finales.
estado_final_p4 = Statevector(np.asarray(resultado_p4.data(0)['salida_final']))
probabilidades_p4 = estado_final_p4.probabilities_dict()
print('\nProbabilidades ideales de medida:')
display(probabilidades_p4)
assert np.isclose(sum(probabilidades_p4.values()), 1.0)
